# Write Economics, Not Glue Code

In pylcm you define small, pure functions — one for utility, one for earnings,
one for each state transition — and pylcm wires them together automatically.
This lets you focus on the economics of your problem rather than on
computational plumbing.

## The idea: parameter-name matching

pylcm (via [dags](https://dags.readthedocs.io/)) inspects function signatures
to build a dependency graph. The rule is simple:

> If function `f` has a **parameter** named `g`, and `g` is also registered
> in the regime (via `functions` or `constraints`), pylcm calls `g` first
> and feeds the result to `f`.

This means function names act as variable names — `labor_income` is both the
name of a function and the name other functions use to request its output.
pylcm walks the resulting directed acyclic graph (DAG) in topological order,
so you never have to think about calling order.

## Example: a consumption-savings model

Consider a model where, each period, the agent chooses labor supply and
consumption. We'll build it up step by step.

First, the imports. We need JAX for numerics, pylcm's grid and model
classes, and pylcm's type aliases (`ContinuousAction`, `DiscreteAction`,
etc.) which document each parameter's role in the model:

In [ ]:
from pprint import pprint

import jax.numpy as jnp

from lcm import (
    AgeGrid,
    DiscreteGrid,
    LinSpacedGrid,
    Model,
    Regime,
    categorical,
)
from lcm.typing import (
    BoolND,
    ContinuousAction,
    ContinuousState,
    DiscreteAction,
    FloatND,
    ScalarInt,
)

Next, we define the discrete outcome spaces. `LaborSupply` enumerates the
agent's labor-supply choices; `RegimeId` enumerates the regimes the agent
can be in. The `@categorical` decorator auto-assigns integer codes starting
from 0:

In [ ]:
@categorical
class LaborSupply:
    work: int
    retire: int


@categorical
class RegimeId:
    working: int
    retired: int
    dead: int

Now the economic functions themselves. Each function is small and
self-contained — it declares exactly what it needs via its signature,
and pylcm takes care of the rest:

In [ ]:
def is_working(labor_supply: DiscreteAction) -> BoolND:
    """Convert the discrete labor-supply action to a boolean."""
    return labor_supply == LaborSupply.work


def labor_income(is_working: BoolND, wage: float | FloatND) -> FloatND:
    """Compute labor income (zero if not working)."""
    return jnp.where(is_working, wage, 0.0)


def utility_working(
    consumption: ContinuousAction, is_working: BoolND, disutility_of_work: float
) -> FloatND:
    """Log utility with a work-disutility term."""
    work_disutility = jnp.where(is_working, disutility_of_work, 0.0)
    return jnp.log(consumption) - work_disutility


def next_wealth(
    wealth: ContinuousState,
    consumption: ContinuousAction,
    labor_income: FloatND,
    interest_rate: float,
) -> ContinuousState:
    """Budget constraint: next-period wealth."""
    return (1 + interest_rate) * (wealth - consumption) + labor_income


def borrowing_constraint(
    consumption: ContinuousAction, wealth: ContinuousState
) -> BoolND:
    """The agent cannot consume more than current wealth."""
    return consumption <= wealth

To use these in a regime, we collect them into `functions` and `constraints`
dicts. The dict **keys** become the names pylcm uses for wiring. Notice
that `utility_working` is registered under the key `"utility"` — pylcm
requires every regime to have a function called `"utility"`, but the
Python function can have any name:

In [ ]:
working_functions = {
    "utility": utility_working,  # key "utility" is required; function can have any name
    "labor_income": labor_income,
    "is_working": is_working,
}

working_constraints = {
    "borrowing_constraint": borrowing_constraint,
}

## The dependency graph

Once we pass these dicts to a `Regime` (see just below), pylcm will
inspect every function in `functions` and `constraints` and see that:

- `is_working` takes `labor_supply` (an action) — no function dependency
- `labor_income` takes `is_working` — must call `is_working` first
- `utility` takes `consumption` (action) and `is_working` — must call
  `is_working` first
- `next_wealth` takes `wealth` (state), `consumption` (action), and
  `labor_income` — must call `labor_income` (and thus `is_working`) first
- `borrowing_constraint` takes `consumption` (action) and `wealth` (state) —
  no function dependency

The resulting DAG looks like this:

```{mermaid}
%%{init: {"theme": "neutral"}}%%
graph LR
    labor_supply --> is_working
    is_working --> labor_income
    is_working --> utility
    labor_income --> next_wealth
    consumption --> utility
    consumption --> next_wealth
    consumption --> borrowing_constraint
    wealth --> next_wealth
    wealth --> borrowing_constraint
```

Notice that `labor_income` appears as both a **function name** and a
**parameter** of `next_wealth`. pylcm resolves this automatically — you
never write `labor_income = labor_income(is_working, wage)` yourself.

## Regime assembly

These functions are bundled into a `Regime` via the `functions` dict — no
manual wiring required:

In [ ]:
def next_regime_from_working(
    labor_supply: DiscreteAction, age: float, final_age_alive: float
) -> ScalarInt:
    return jnp.where(
        age >= final_age_alive,
        RegimeId.dead,
        jnp.where(
            labor_supply == LaborSupply.retire,
            RegimeId.retired,
            RegimeId.working,
        ),
    )


def next_regime_from_retired(age: float, final_age_alive: float) -> ScalarInt:
    return jnp.where(age >= final_age_alive, RegimeId.dead, RegimeId.retired)


def next_wealth_retired(
    wealth: ContinuousState,
    consumption: ContinuousAction,
    interest_rate: float,
) -> ContinuousState:
    """Budget constraint for retirees (no labor income)."""
    return (1 + interest_rate) * (wealth - consumption)


def utility_retired(consumption: ContinuousAction) -> FloatND:
    return jnp.log(consumption)


ages = AgeGrid(start=60, stop=85, step="5Y")

working = Regime(
    transition=next_regime_from_working,
    active=lambda age: age < ages.exact_values[-1],
    states={
        "wealth": LinSpacedGrid(start=1, stop=400, n_points=100),
    },
    state_transitions={
        "wealth": next_wealth,
    },
    actions={
        "labor_supply": DiscreteGrid(LaborSupply),
        "consumption": LinSpacedGrid(start=1, stop=400, n_points=500),
    },
    functions=working_functions,
    constraints=working_constraints,
)

retired = Regime(
    transition=next_regime_from_retired,
    active=lambda age: age < ages.exact_values[-1],
    states={
        "wealth": LinSpacedGrid(start=1, stop=400, n_points=100),
    },
    state_transitions={
        "wealth": next_wealth_retired,
    },
    actions={"consumption": LinSpacedGrid(start=1, stop=400, n_points=500)},
    functions={"utility": utility_retired},
    constraints={"borrowing_constraint": borrowing_constraint},
)

dead = Regime(
    transition=None,
    active=lambda age: age >= ages.exact_values[-1],
    functions={"utility": lambda: 0.0},
)

The `functions` and `constraints` dicts are all pylcm needs. It inspects
the signatures of every registered function and constraint, builds the
dependency graph, and determines that `is_working` must run before
`labor_income`, which must run before `next_wealth`.

This is the correct mental model for what pylcm does internally: it
collects all the functions you provide (via `functions`, `constraints`,
and `state_transitions`), resolves their dependencies by matching
parameter names to function names, and executes them in the right order.
You describe the economics; pylcm handles the plumbing.

## What you don't have to do

Compare this with what a manual approach would require:

- **No need to call functions in the right order.** pylcm topologically
  sorts the DAG for you.
- **No need to pass intermediate results between functions.** The output
  of `is_working` is automatically fed to `labor_income` and
  `utility_working`.
- **No need to manage a shared state dict.** Each function declares
  exactly what it needs via its signature.
- **Adding a new auxiliary function is trivial.** Just add it to the
  `functions` dict and use its name as a parameter in other functions.

## Parameters

Parameters that aren't provided by other functions — like `wage`,
`interest_rate`, and `disutility_of_work` — become model parameters.
Use `model.get_params_template()` to see the expected structure:

In [ ]:
model = Model(
    regimes={"working": working, "retired": retired, "dead": dead},
    ages=ages,
    regime_id_class=RegimeId,
)

pprint(model.get_params_template())

The template mirrors the function structure: each regime lists the
functions that need external parameters, and each function lists the
parameter names that aren't satisfied by other functions or by
states/actions.

## Solve and simulate

In [ ]:
params = {
    "discount_factor": 0.95,
    "interest_rate": 0.05,
    "final_age_alive": ages.exact_values[-2],
    "working": {
        "utility": {"disutility_of_work": 0.5},
        "labor_income": {"wage": 10.0},
    },
}

n_agents = 50

result = model.solve_and_simulate(
    params=params,
    initial_regimes=["working"] * n_agents,
    initial_states={
        "age": jnp.full(n_agents, ages.exact_values[0]),
        "wealth": jnp.linspace(10, 300, n_agents),
    },
)

df = result.to_dataframe(additional_targets="all")
df.head(10)